<a href="https://colab.research.google.com/github/prithwis/Centaur/blob/main/MOS_K_Means.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from google.colab import files
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import adjusted_rand_score, silhouette_score
from sklearn.decomposition import PCA




In [4]:
df = pd.read_csv("MOS_TR1ME3_W3_Cluster_Data.csv")

print("Shape:", df.shape)
print("Columns:", df.columns.tolist())

display(df.head())

Shape: (20, 11)
Columns: ['Run_ID', 'Cluster_ID', 'Cluster_Name', 'Oil Price', 'Hormuz Oil Flow', 'Regional Diplomatic Engagement', 'Economic Pressure on Iran', 'US/Regional Partner Cohesion', 'Iran Domestic Stability', 'US-Iran Tension', 'Regional Armed Conflict']


,Run_ID,Cluster_ID,Cluster_Name,Oil Price,Hormuz Oil Flow,Regional Diplomatic Engagement,Economic Pressure on Iran,US/Regional Partner Cohesion,Iran Domestic Stability,US-Iran Tension,Regional Armed Conflict
0,MOS_TR1ME3_R01,C2,"Military Action, Diplomatic Response",111,83,101,109,108,94,116,119
1,MOS_TR1ME3_R02,C1,Military Confrontation,118,84,86,105,110,92,124,122
2,MOS_TR1ME3_R03,C2,"Military Action, Diplomatic Response",115,75,94,103,110,94,121,120
3,MOS_TR1ME3_R04,C2,"Military Action, Diplomatic Response",114,79,96,110,108,92,115,115
4,MOS_TR1ME3_R05,C1,Military Confrontation,119,70,85,108,113,94,125,124


In [5]:
# First three columns: identifiers and narrative classification
labels = df.iloc[:, :3].copy()

# Remaining eight columns: numerical WorldState variables
X = df.iloc[:, 3:].copy()

# Ensure all eight variables are numeric
X = X.apply(pd.to_numeric, errors="raise")

print("Narrative labels:", labels.columns.tolist())
print("WorldState variables:", X.columns.tolist())
print("Shape of X:", X.shape)

display(X.head())

Narrative labels: ['Run_ID', 'Cluster_ID', 'Cluster_Name']
WorldState variables: ['Oil Price', 'Hormuz Oil Flow', 'Regional Diplomatic Engagement', 'Economic Pressure on Iran', 'US/Regional Partner Cohesion', 'Iran Domestic Stability', 'US-Iran Tension', 'Regional Armed Conflict']
Shape of X: (20, 8)


,Oil Price,Hormuz Oil Flow,Regional Diplomatic Engagement,Economic Pressure on Iran,US/Regional Partner Cohesion,Iran Domestic Stability,US-Iran Tension,Regional Armed Conflict
0,111,83,101,109,108,94,116,119
1,118,84,86,105,110,92,124,122
2,115,75,94,103,110,94,121,120
3,114,79,96,110,108,92,115,115
4,119,70,85,108,113,94,125,124


In [6]:

from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_scaled = scaler.fit_transform(X)

# Convert back to DataFrame for inspection
XS = pd.DataFrame(X_scaled, columns=X.columns)

display(XS.head().round(2))


,Oil Price,Hormuz Oil Flow,Regional Diplomatic Engagement,Economic Pressure on Iran,US/Regional Partner Cohesion,Iran Domestic Stability,US-Iran Tension,Regional Armed Conflict
0,-0.65,0.52,1.25,1.06,-0.48,0.39,-0.61,-0.06
1,0.75,0.56,-0.58,-0.09,0.07,-0.23,0.58,0.49
2,0.15,0.12,0.40,-0.66,0.07,0.39,0.13,0.12
3,-0.05,0.32,0.64,1.35,-0.48,-0.23,-0.76,-0.81
4,0.96,-0.13,-0.70,0.78,0.88,0.39,0.73,0.86


In [7]:

from sklearn.cluster import KMeans

kmeans = KMeans(n_clusters=3, random_state=42, n_init=20)

clusters = kmeans.fit_predict(XS)

# Display cluster assignment for each run
results = pd.DataFrame({
    "Run_ID": df.iloc[:, 0],
    "Numerical_Cluster": clusters
})

display(results)

print("\nCluster frequencies:")
print(results["Numerical_Cluster"].value_counts().sort_index())


,Run_ID,Numerical_Cluster
0,MOS_TR1ME3_R01,1
1,MOS_TR1ME3_R02,0
2,MOS_TR1ME3_R03,0
3,MOS_TR1ME3_R04,1
4,MOS_TR1ME3_R05,2
5,MOS_TR1ME3_R06,2
6,MOS_TR1ME3_R07,2
7,MOS_TR1ME3_R08,1
8,MOS_TR1ME3_R09,2
9,MOS_TR1ME3_R10,2



Cluster frequencies:
Numerical_Cluster
0    5
1    8
2    7
Name: count, dtype: int64


In [8]:

# Narrative classification from the CSV (second column)
narrative = df.iloc[:, 1]

# Cross-tabulation
comparison = pd.crosstab(
    narrative,
    clusters,
    rownames=["Narrative Cluster"],
    colnames=["Numerical Cluster"]
)

display(comparison)


Numerical Cluster,0,1,2
Narrative Cluster,,,
C1,3,0,7
C2,2,5,0
C3,0,3,0


In [9]:

from sklearn.metrics import adjusted_rand_score

ari = adjusted_rand_score(narrative, clusters)

print(f"Adjusted Rand Index (ARI): {ari:.3f}")


Adjusted Rand Index (ARI): 0.389


In [10]:

# Average original W3 indices within each numerical cluster
centroids = X.groupby(clusters).mean().round(1)

centroids.index.name = "Numerical Cluster"

display(centroids.T)


Numerical Cluster,0,1,2
Oil Price,113.8,111.1,118.1
Hormuz Oil Flow,77.8,79.9,60.7
Regional Diplomatic Engagement,90.4,98.8,81.9
Economic Pressure on Iran,102.4,104.9,107.9
US/Regional Partner Cohesion,109.8,106.6,113.3
Iran Domestic Stability,92.4,95.4,90.0
US-Iran Tension,121.6,113.0,127.1
Regional Armed Conflict,120.6,113.8,124.9


In [11]:

from sklearn.metrics import silhouette_score

silhouette = silhouette_score(XS, clusters)

print(f"Silhouette Score: {silhouette:.3f}")


Silhouette Score: 0.198


In [12]:

from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

for k in range(2, 7):
    model = KMeans(n_clusters=k, random_state=42, n_init=20)
    predicted = model.fit_predict(XS)

    score = silhouette_score(XS, predicted)

    print(f"k = {k}  |  Silhouette = {score:.3f}")


k = 2  |  Silhouette = 0.346
k = 3  |  Silhouette = 0.198
k = 4  |  Silhouette = 0.203
k = 5  |  Silhouette = 0.227
k = 6  |  Silhouette = 0.221


In [2]:
# -------------------------------------------------------
# 1. Upload both CSV files
# -------------------------------------------------------

#uploaded = files.upload()

# -------------------------------------------------------
# 2. Analyse one scenario
# -------------------------------------------------------

def analyseClusters(filename):

    df = pd.read_csv(filename)

    # Eight WorldState variables
    feature_cols = [
        c for c in df.columns
        if c not in ["Run_ID", "Group_ID", "Group_Name"]
    ]

    X = df[feature_cols].apply(pd.to_numeric, errors="raise")

    # Standardize the eight numerical variables
    X_scaled = StandardScaler().fit_transform(X)

    # Numerical clustering, without narrative labels
    model = KMeans(n_clusters=3, random_state=42, n_init=20)
    numerical_labels = model.fit_predict(X_scaled)

    # Compare against narrative classification
    narrative_labels = df["Group_ID"]
    ari = adjusted_rand_score(narrative_labels, numerical_labels)
    silhouette = silhouette_score(X_scaled, numerical_labels)

    print("\n" + "=" * 65)
    print("SCENARIO:", filename)
    print("=" * 65)
    print("Runs:", len(df))
    print("WorldState variables:", len(feature_cols))
    print("Adjusted Rand Index:", round(ari, 3))
    print("Silhouette Score:", round(silhouette, 3))

    print("\nNarrative vs Numerical Clusters")
    print(pd.crosstab(
        narrative_labels,
        numerical_labels,
        rownames=["Narrative"],
        colnames=["K-Means"]
    ))

    # Individual run assignments
    results = df[["Run_ID", "Group_ID", "Group_Name"]].copy()
    results["KMeans_Cluster"] = numerical_labels

    print("\nRun-by-run classification")
    display(results.sort_values("Run_ID"))

    # PCA visualization (for display only)
    pca = PCA(n_components=2)
    points = pca.fit_transform(X_scaled)

    fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

    axes[0].scatter(
        points[:, 0], points[:, 1],
        c=pd.factorize(narrative_labels)[0],
        cmap="viridis", s=75
    )
    axes[0].set_title("Narrative Clusters")

    axes[1].scatter(
        points[:, 0], points[:, 1],
        c=numerical_labels,
        cmap="viridis", s=75
    )
    axes[1].set_title("K-Means Clusters")

    for ax in axes:
        for i, run in enumerate(df["Run_ID"]):
            ax.annotate(str(run).split("_")[-1],
                        (points[i, 0], points[i, 1]),
                        fontsize=7, xytext=(3, 3),
                        textcoords="offset points")
        ax.set_xlabel("Principal Component 1")
        ax.set_ylabel("Principal Component 2")

    plt.suptitle(filename)
    plt.tight_layout()
    plt.show()

    print("PCA variance explained:",
          round(pca.explained_variance_ratio_.sum() * 100, 1), "%")

    return results




In [3]:
# -------------------------------------------------------
# 3. Run both experiments independently
# -------------------------------------------------------

TR1 = analyseClusters("MOS_TR1ME3_W3_Cluster_Data.csv")
#TR2 = analyseClusters("MOS_TR2ME3_W3_Cluster_Data.csv")

ValueError: Unable to parse string "C2" at position 0